In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQFAM_F.xpt"
csv_file = DATA_DIR / "SMQFAM_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched source values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 SMQFAM_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

#
# CDC 2009-2010 SMQFAM_F:
#
# SEQN
# SMD410
# SMD415
# SMD415A
# SMD430
#

EXPECTED_COLUMNS = [
    "SEQN",
    "SMD410",
    "SMD415",
    "SMD415A",
    "SMD430"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQFAM_F column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 10537:

    raise ValueError(
        f"Unexpected row count: {len(df):,}. "
        "Expected 10,537."
    )


if len(df.columns) != 5:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,537 rows x 5 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Never globally replace this value.
#
# For SMQFAM_F, CDC does NOT publish zero as a valid
# released value for:
#
#     SMD410
#     SMD415
#     SMD415A
#     SMD430
#
# Therefore the expected tiny count is ZERO.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts found "
        "in SMQFAM_F. CDC does not indicate legitimate "
        "zero values in the released variables, so do NOT "
        "automatically replace these values."
    )


if true_zero_total != 0:

    print(
        "\nTrue zeros by variable:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected true numeric zero values "
        "found in SMQFAM_F."
    )


print(
    "PASS: No tiny XPORT zero artifacts."
)

print(
    "PASS: No unexpected numeric zeros."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimals.
#
# Always check BEFORE converting values to Int64.
#
# SMQFAM_F contains questionnaire counts/codes only,
# so genuine decimal variables are expected to be NONE.
#
# This same safeguard protects genuine decimal fields
# in Dietary, Labs, Examination, etc.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in SMQFAM_F."
)


# ============================================================
# 6. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "SEQN contains missing values."
    )


if (
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "SEQN contains duplicates."
    )


if df["SEQN"].min() != 51624:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 62160:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# All five numeric variables have now been proven
# to contain whole-number values only.
#
# Therefore convert to nullable Int64.
#
# IMPORTANT:
#
# NO .round() IS USED.
#

INTEGER_COLUMNS = [
    "SEQN",
    "SMD410",
    "SMD415",
    "SMD415A",
    "SMD430"
]


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        if not np.all(
            np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All SMQFAM_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 8. CDC CHECK - SMD410
# ============================================================

#
# Does anyone smoke inside home?
#
# CDC:
#
# 1 = Yes          1,517
# 2 = No           8,948
# 7 = Refused          3
# 9 = Don't know       2
# Missing             67
#

EXPECTED_SMD410 = {
    1: 1517,
    2: 8948,
    7: 3,
    9: 2
}


actual = (
    df["SMD410"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- SMD410 ---")

print(
    actual
)


if actual != EXPECTED_SMD410:

    raise ValueError(
        "SMD410 frequencies differ from CDC.\n"
        f"Expected: {EXPECTED_SMD410}\n"
        f"Actual:   {actual}"
    )


if int(
    df["SMD410"]
    .isna()
    .sum()
) != 67:

    raise ValueError(
        "SMD410 missing count differs from CDC."
    )


print(
    "PASS: SMD410 matches CDC."
)


# ============================================================
# 9. CDC CHECK - SMD415
# ============================================================

#
# Total number of smokers inside home.
#
# CDC:
#
# 1-2 = 1,380
# 3   = 137
# Missing = 9,020
#

s = df["SMD415"]


print("\n--- SMD415 ---")


print(
    "1-2:",
    int(
        s.between(
            1,
            2
        ).sum()
    )
)


print(
    "3 or more:",
    int(
        s.eq(3)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            2
        ).sum()
    )
    == 1380
    and
    int(
        s.eq(3)
        .sum()
    )
    == 137
    and
    int(
        s.isna()
        .sum()
    )
    == 9020
):

    raise ValueError(
        "SMD415 differs from CDC."
    )


print(
    "PASS: SMD415 matches CDC."
)


# ============================================================
# 10. CDC CHECK - SMD415A
# ============================================================

#
# Total number of cigarette smokers inside home.
#
# CDC:
#
# 1-2 = 1,315
# 3   = 126
# Missing = 9,096
#

s = df["SMD415A"]


print("\n--- SMD415A ---")


print(
    "1-2:",
    int(
        s.between(
            1,
            2
        ).sum()
    )
)


print(
    "3 or more:",
    int(
        s.eq(3)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            2
        ).sum()
    )
    == 1315
    and
    int(
        s.eq(3)
        .sum()
    )
    == 126
    and
    int(
        s.isna()
        .sum()
    )
    == 9096
):

    raise ValueError(
        "SMD415A differs from CDC."
    )


print(
    "PASS: SMD415A matches CDC."
)


# ============================================================
# 11. CDC CHECK - SMD430
# ============================================================

#
# Total cigarettes per day smoked inside home.
#
# CDC instructions:
#
#   1 pack = 20 cigarettes
#   If none, enter 0
#   If less than 1 per day, enter 1
#
# BUT in the released public data:
#
# 1-39 = 1,277
# 40   =   164
# Missing = 9,096
#
# No released zero values occur in this file.
#

s = df["SMD430"]


print("\n--- SMD430 ---")


print(
    "1-39:",
    int(
        s.between(
            1,
            39
        ).sum()
    )
)


print(
    "40 or more:",
    int(
        s.eq(40)
        .sum()
    )
)


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            39
        ).sum()
    )
    == 1277
    and
    int(
        s.eq(40)
        .sum()
    )
    == 164
    and
    int(
        s.eq(0)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 9096
):

    raise ValueError(
        "SMD430 differs from CDC."
    )


print(
    "PASS: SMD430 matches CDC."
)


# ============================================================
# 12. LOGICAL CONSISTENCY CHECK
# ============================================================

#
# SMD415A = total cigarette smokers
# SMD415  = total smokers of all tobacco types
#
# Therefore cigarette-smoker count should never
# exceed total-smoker count where both are present.
#

both_present = (
    df["SMD415"]
    .notna()
    &
    df["SMD415A"]
    .notna()
)


bad_smoker_count = int(
    (
        df.loc[
            both_present,
            "SMD415A"
        ]
        >
        df.loc[
            both_present,
            "SMD415"
        ]
    )
    .sum()
)


print("\n--- LOGICAL CONSISTENCY ---")


print(
    "Cigarette smokers > total smokers:",
    bad_smoker_count
)


if bad_smoker_count != 0:

    raise ValueError(
        "SMD415A exceeds SMD415 for one or more records."
    )


print(
    "PASS: Cigarette-smoker counts are logically consistent."
)


# ============================================================
# 13. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# SMQFAM_F requires NO numeric correction.
#
# Therefore every value must remain exactly equal
# to the source XPT.
#
# Int64 changes CSV representation only:
#
#     51624.0 -> 51624
#     2.0     -> 2
#     20.0    -> 20
#
# It does not change the numeric value.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 5 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 14. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 15. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 16. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 5 CSV columns match "
    "source XPT values exactly."
)


# ============================================================
# 17. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 18. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny-value artifacts."
)


# ============================================================
# 19. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# SMQFAM_F contains NO genuine decimal variables.
#
# Therefore ".0" should not appear anywhere in this
# particular CSV.
#
# IMPORTANT:
#
# Do NOT use this global test on Dietary or mixed
# continuous-measurement datasets.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 20. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 21. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "SMQFAM_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "True numeric zeros:",
    true_zero_total
)


print(
    "Tiny-value corrections: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 SMQFAM_F XPT -> CSV
Rows: 10,537
Columns: 5
PASS: Structure = 10,537 rows x 5 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero artifacts.
PASS: No unexpected numeric zeros.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in SMQFAM_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All SMQFAM_F fields formatted without unnecessary .0.

--- SMD410 ---
{np.int64(1): 1517, np.int64(2): 8948, np.int64(7): 3, np.int64(9): 2}
PASS: SMD410 matches CDC.

--- SMD415 ---
1-2: 1380
3 or more: 137
Missing: 9020
PASS: SMD415 matches CDC.

--- SMD415A ---
1-2: 1315
3 or more: 126
Missing: 9096
PASS: SMD415A matches CDC.

--- SMD430 ---
1-39: 1277
40 or more: 164
Zero: 0
Missing: 9096
PASS: SMD430 matches CDC.

--- LOGICAL CONSISTENCY ---
Cigarette smokers > total smokers: 0
PASS: Cigarette-smoker counts are logically consistent.

--- VALUE FI